# 02 · Knowledge Tracing — BKT vs SAKT

Goal: model a learner's **mastery over time** and predict whether they'll get the *next*
question right. Metric: next-question **AUC** on a group-aware (by-learner) split.

- **BKT** — one interpretable 2-state HMM per skill (baseline, `reflecta.models.knowledge_tracing`).
- **SAKT** — a self-attention block that lets the prediction attend over the learner's whole
  history, capturing cross-skill transfer BKT can't (`reflecta.models.sakt`).

The heavy lifting lives in the package; this notebook is the research narrative. Full runs:
`python scripts/run_pipeline.py` (IRT+BKT) and `python scripts/train_sakt.py` (SAKT).

In [ ]:
import pandas as pd
from reflecta.config import CONFIG
from reflecta.pipeline import ingest, clean

# load the processed ASSISTments written by the pipeline (or ingest fresh)
parquet = CONFIG.paths.processed / 'assistments_clean.parquet'
if parquet.exists():
    df = pd.read_parquet(parquet)
else:
    raw, src = ingest('auto'); df = clean(raw, src)

# subsample whole learners so sequences stay intact (keeps the notebook fast)
keep = df['learner_id'].drop_duplicates().head(1200)
sample = df[df['learner_id'].isin(keep)].reset_index(drop=True)
print(f'{len(sample):,} interactions · {sample.learner_id.nunique()} learners · {sample.skill.nunique()} skills')

## BKT baseline

Fit one BKT per skill on the train split, evaluate next-question AUC on held-out learners.
(The full-data pipeline reports **BKT val AUC ≈ 0.763**.)

In [ ]:
import numpy as np
from reflecta.data.split import group_train_val_split
from reflecta.models.knowledge_tracing import BKT
from reflecta.eval.metrics import next_question_auc

tr, va = group_train_val_split(sample, group_col='learner_id', val_frac=0.2)

def seqs(d, skill):
    s = d[d.skill == skill]
    return [g.tolist() for _, g in s.groupby('learner_id')['correct'] if len(g) >= 2]

top = tr.skill.value_counts().head(10).index
preds, true = [], []
for sk in top:
    bkt = BKT().fit_grid(seqs(tr, sk), grid=5)
    for s in seqs(va, sk):
        preds += bkt.predict_sequence(s); true += s
print('BKT val AUC:', round(next_question_auc(np.array(preds), np.array(true)), 4))

## SAKT (self-attentive KT)

A few epochs on this sample; scale up with `scripts/train_sakt.py` on GPU (Kaggle/Colab).

In [ ]:
from reflecta.models.sakt import train_sakt

result = train_sakt(sample, max_len=100, d_model=64, epochs=5)
print('SAKT best val AUC:', round(result['val_auc'], 4))

## Read-out & next steps

| Model | Val next-question AUC (full data) | Notes |
|-------|-----------------------------------|-------|
| IRT 1PL | ~0.51 | near chance on held-out learners — can't cold-start (see `models/online_irt.py`) |
| BKT | **~0.763** | interpretable, strong baseline |
| SAKT | scale up on GPU | self-attention over full history; published ~0.75–0.85 |

Directions: (1) per-*item* SAKT (exercise ids, not just skills); (2) add a forgetting/time
feature from `response_time` gaps; (3) join with **Eedi** misconception labels so wrong-answer
attention becomes *diagnostic*, feeding Reflecta signal #2.